# Simulated Annealing for Vehicle Routing

We assign customers to routes and choose their visiting order, respecting capacity, service windows and depot return. All experiments minimize **total distance only**, using arcs truncated to one decimal and travel time equal to distance. Vehicle count is reported without a penalty.

The main study uses **all 56 supplied Solomon instances**, with ten SA seeds per instance: **560 SA runs**. It reports per-instance means, sample standard deviations, iteration counts and elapsed times. Each run receives 20 seconds including construction, with independent final feasibility validation. Two smaller diagnostic experiments remain available separately.


## Setup and code structure

Run cells from top to bottom. Install the project with `pip install -e ".[dev]"` if needed. Numba is a main dependency; its absence is an installation error rather than a reason to silently skip SA tests.

The implementation lives in `src/cvrp/sa_vrptw.py` (data model, construction, evaluator and study helpers) and `src/cvrp/sa_static.py` (compiled moves). Tests import these modules independently of notebook cell positions.

Independent SA runs use up to six processes, bounded by CPU count. Each run remains single-threaded with its own 20-second budget. Each worker warms up Numba on synthetic data before timed work. The main study contains 560 runs: about 187 minutes sequentially, or roughly 31 minutes with six workers, plus setup and diagnostics. A seed fixes random choices, but wall-clock stopping can produce different move counts on different machines.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import os
import platform
import sys
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data' / 'solomon'
TIME_LIMIT = 20.0
SEEDS = tuple(range(10))
WORKERS = min(6, max(1, (os.cpu_count() or 2) - 1))
print('Python:', sys.version.split()[0], '| Machine:', platform.platform())
print({name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'vrplib', 'numba')})
print(f'{WORKERS} workers, {len(SEEDS)} seeds, {TIME_LIMIT:.0f} seconds per run')


Python: 3.14.7 | Machine: Windows-11-10.0.26300-SP0
{'numpy': '2.5.3', 'pandas': '3.0.6', 'vrplib': '2.2.0', 'numba': '0.68.0'}
6 workers, 10 seeds, 20 seconds per run


## Local Solomon data and reference objective

We use **every supplied Solomon `.txt` instance and accompanying `.sol` reference** in git-ignored `data/solomon/`. The older `.vrp` copies are not read or downloaded. SHA256 hashes identify the input bytes.

The supplied references are distance-minimizing solutions, not fleet-first solutions. R101 has 20 reference routes and cost 1,637.7; the separate fleet-first solution uses 19 routes and costs 1,650.8. We check coverage, fleet, capacity, time windows and recomputed truncated-distance costs. Reference routes never initialize or guide SA.

The study includes all six families (C1, C2, R1, R2, RC1, RC2) and all 56 instances, including C101, C201, R101, R201, RC101 and RC201. No tuning-instance exclusion or 24-instance selection is applied. This is a full-dataset descriptive study, not a held-out evaluation. C101-25 and RC101 are also available as separate diagnostics.


In [2]:
from cvrp.sa_vrptw import solomon_names, load_solomon

INSTANCES = solomon_names(DATA)
assert len(INSTANCES) == 56, f'Expected the 56 supplied Solomon instances, found {len(INSTANCES)}'
for name in INSTANCES:
    for extension in ('txt', 'sol'):
        path = DATA / f'{name}.{extension}'
        print(path.name, '| SHA256:', hashlib.sha256(path.read_bytes()).hexdigest())
    load_solomon(DATA, name)
print(f'Validated all {len(INSTANCES)} instance/reference pairs with truncated distances.')


C101.txt | SHA256: a6da75152d182d60ecd2c6f854296f5be452f92282d096adebcf5d99a7f16516
C101.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8973be0f00
C102.txt | SHA256: f89a0e2a760cc7448a4ca72b88e5b29b94d64b5548f3320f42a6230025ead653
C102.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8973be0f00
C103.txt | SHA256: a55cabf15e0da9d65696a5fb45c9852889479893f76f772a428ca15d1b415a51
C103.sol | SHA256: 675b45a47a6f1433bc9606349ca4baafb0ba255248ca20a68c6b3495d2dba17b
C104.txt | SHA256: fe333e81625ee5a75bb2ce98c2c9f356d07289237865b62a23ba2f32f0de5ee2
C104.sol | SHA256: c9759507f943c984c4a2475966d1058afc42a2158bd75f5052461fc2fc552e15
C105.txt | SHA256: 9a877a3a6fb922e7ad9297cc0308bc8a5652cb1495a0228ee6b81b2d938aa29f
C105.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8973be0f00
C106.txt | SHA256: ec9b419bed53e9c607cce76e8c6bc59feb6e4d17646905601f008a4141ec7b40
C106.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce897

## Solution representation and feasibility

Customers are `1, ..., n`; `0` is the depot. A solution `[[1, 4], [2, 3]]` represents routes `0 -> 1 -> 4 -> 0` and `0 -> 2 -> 3 -> 0`. Depot visits are implicit.

`make_problem` extracts coordinates, demands, service times, windows and fleet limit. Arcs use `floor(10 * Euclidean distance) / 10`, both for distance and travel time. This matches Tabu and the supplied references. Exact distances remain an explicit library option, but are not used here.

`simulate` travels between customers, waits for opening times and adds service durations. Service must begin before each closing time, and return must respect the depot window. Demand must fit capacity. `validate` also checks coverage and fleet size, recomputing schedules independently of the compiled kernel. Every returned solution is checked.

Traffic mode and fleet-first benchmark mode remain library options for separate studies. Neither is selected below.


In [3]:
from cvrp.sa_vrptw import Problem, make_problem, simulate, evaluate_route, validate


## Why Simulated Annealing?

A move changes customer assignments or visit order. Accepting only improvements can trap the search in a local optimum: every available small change looks worse, even though several changes together could improve the solution.

SA also accepts some worsening moves. Let $E$ be the selected objective and $\Delta = E_{new} - E_{current}$:

- If $\Delta \leq 0$, accept the move.
- Otherwise, accept it with probability $\exp(-\Delta/T)$, where $T$ is the temperature.

Higher temperatures allow more exploration; lower temperatures make acceptance more selective. Infeasible candidates are rejected regardless of temperature. The algorithm keeps a separate **best feasible solution**, so accepting a worse current solution never loses the best result already found.

SA explores only part of the search space. It does not prove optimality or guarantee a particular gap on other instances or hardware.


## Initial solution and neighborhood moves

`initialize` builds a solution without reference information. `repair` shuffles customers and inserts each into a feasible position. It considers up to eight nearby existing routes, usually chooses the cheapest insertion, and sometimes selects another good insertion to diversify construction. A new route is opened when no considered insertion is feasible and the fleet limit permits it. Failed construction can be retried within the time budget.

The compiled static search then samples five move types:

| Move | What changes |
| --- | --- |
| Relocate | Move one customer to another route |
| Swap | Exchange two customers, within one route or between routes |
| Segment reversal | Reverse a sequence within a route (2-opt) |
| Short-chain relocation | Move a consecutive block of two or three customers to another route |
| Route-tail exchange | Exchange the ends of two routes (2-opt-star), which can also merge routes |

Empty routes do not count as vehicles. Every proposed move must preserve capacity and time-window feasibility before the SA acceptance rule is applied.

The Python fallback instead combines destroy-and-repair, swaps and reversals. The `repair` function below is used for initialization by the compiled solver as well as for moves by that fallback.


In [4]:
from cvrp.sa_vrptw import repair, initialize
# Shared construction and evaluation are imported, not copied into the notebook.


## Objective, parameter provenance and timing

The objective is `E = total distance`. A new vehicle is allowed if it reduces distance within the fleet limit. The optional fleet-first library mode is not used in this comparison.

Compiled SA uses `T0 = max(1, 0.005 * initial_distance)` and `Tf = 0.02`. Temperature falls exponentially with elapsed time: `T(t) = T0 * (Tf / T0) ** (t / time_limit)`. The initial factor scales temperature to the constructed solution; the final temperature makes worsening moves increasingly unlikely near the deadline.

**These are fixed heuristic defaults, not the result of a tuning experiment on a separate tuning set.** They were adopted during implementation and exploratory runs, including these comparison instances. Accordingly, this is not a pristine validation set for SA parameter selection, and no claim of optimal parameters is made. Both settings are configurable in `anneal_static` for future tuning. All 56 instances are included here, so this study makes no held-out validation claim.

The search budget includes construction. SA checks its deadline between batches of 256 move attempts. `elapsed_s` also includes independent final validation, so a small overrun is possible. Synthetic Numba compilation is excluded as setup. The Python fallback's `alpha` and minimum-temperature settings do not control compiled static cooling.


In [5]:
from cvrp.sa_static import anneal_static, warmup
from cvrp.sa_vrptw import anneal, run_experiment, run_comparison, summarize_runs
warmup()  # Synthetic compilation, outside the timed experiments.


## Reference checks

The local `.sol` files provide distance-oriented references. `load_solomon` validates routes with truncated arcs and checks that their recomputed cost agrees with the file. Invalid references raise an error rather than silently losing their gaps.

All 56 supplied references are valid, including R102, R105 and RC105. Their previous missing gaps came from evaluating truncated-distance reference routes with exact travel times.

C101-25 uses the C101 reference restricted to customers 1-25, removing empty routes. This is a comparison route set, not a certified optimum for the reduced instance.


In [6]:
p25, ref25 = load_solomon(DATA, 'C101', n=25)
pRC, refRC = load_solomon(DATA, 'RC101')
assert np.isclose(validate(p25, ref25)['distance'], 191.3)
for p, routes in ((p25, ref25), (pRC, refRC)):
    metrics = validate(p, routes)
    print(f"{p.name}: {p.n} customers, {metrics['vehicles']} reference routes, "
          f"distance {metrics['distance']:.1f}, feasible")


C101-25: 25 customers, 3 reference routes, distance 191.3, feasible
RC101-100: 100 customers, 15 reference routes, distance 1619.8, feasible


## Reading gaps, variability and work counts

For each SA run, `gap_pct = 100 * (SA_distance / reference_distance - 1)`. Positive values mean longer distance, zero means equal distance, and negative values mean shorter distance. No gap is capped or filtered.

We report **mean and sample standard deviation (`ddof=1`) across ten seeds for each instance**. Standard deviation measures run-to-run variability, not a confidence interval. The raw `comparison` table retains every seed's distance, gap, fleet size, elapsed time and iteration count.

| Field | Meaning |
| --- | --- |
| `elapsed_s` | Total seconds, including construction, search and final validation |
| `iterations` | Proposed SA move attempts, including proposals rejected early |
| `iterations_per_s` | Move-attempt throughput |
| `engine` | `numba`, the compiled SA engine |
| `reference_feasible`, `feasible` | Independent reference and result checks |

The fixed time budget measures SA together with its implementation speed. Move attempts are not all equally expensive, so iteration counts provide context rather than a direct measure of algorithmic complexity. This notebook runs SA only.


## Small-instance experiment: C101-25

Run ten SA seeds (0-9), with distance-only optimization, truncated distances and a 20-second budget each. Independent runs can use separate workers; every run constructs its own feasible start.

The raw table includes iterations for each seed. The summary reports mean and sample standard deviation of distance, gap, time and iterations. Results remain in memory and are displayed; the notebook does not save CSV files.


In [7]:
results25, runs25 = run_experiment(
    p25, ref25, SEEDS, time_limit=TIME_LIMIT, workers=WORKERS
)
display(results25[['seed', 'vehicles', 'distance', 'gap_pct', 'elapsed_s', 'iterations']])
display(summarize_runs(results25).round(3))
best25 = min(runs25, key=lambda out: out['metrics']['distance'])
print('Best routes:', best25['routes'])


,seed,vehicles,distance,gap_pct,elapsed_s,iterations
0,0,3,191.3,0.0,20.000191,61533952
1,1,3,191.3,0.0,20.000197,59365120
2,2,3,191.3,0.0,20.000238,58179840
3,3,3,191.3,0.0,20.000150,57675008
4,4,3,191.3,0.0,20.000144,56242944
5,5,3,191.3,0.0,20.000120,55927552
6,6,3,191.3,0.0,20.000274,63230720
7,7,3,191.3,0.0,20.000286,59434240
8,8,3,191.3,0.0,20.000185,60736256
9,9,3,191.3,0.0,20.000100,58574336


,instance,algorithm,runs,distance_mean,distance_std,gap_mean,gap_std,elapsed_mean,elapsed_std,iterations_mean,iterations_std
0,C101-25,sa,10,191.3,0.0,0.0,0.0,20.0,0.0,59089996.8,2290708.834


Best routes: [[5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1], [13, 17, 18, 19, 15, 16, 14, 12], [20, 24, 25, 23, 22, 21]]


## Full-instance diagnostic: RC101

Repeat the same distance-only, truncated-distance protocol on 100 customers, using ten SA seeds and 20 seconds per run. Inspect both variability and iteration counts rather than assuming equal work per elapsed second. The next observation cells generate their text directly from the diagnostic tables.


In [8]:
rc_results, rc_runs = run_experiment(
    pRC, refRC, SEEDS, time_limit=TIME_LIMIT, workers=WORKERS
)
display(rc_results[['seed', 'vehicles', 'distance', 'gap_pct', 'elapsed_s', 'iterations']])
display(summarize_runs(rc_results).round(3))


,seed,vehicles,distance,gap_pct,elapsed_s,iterations
0,0,17,1640.9,1.302630,20.000370,54012672
1,1,16,1659.0,2.420052,20.000319,65700608
2,2,18,1659.4,2.444746,20.000368,72993792
3,3,16,1642.3,1.389060,20.000597,55412992
4,4,17,1655.0,2.173108,20.000567,51780352
5,5,16,1634.2,0.888999,20.000308,69497600
6,6,16,1634.4,0.901346,20.000295,63445504
7,7,17,1650.1,1.870601,20.000306,80915200
8,8,16,1637.6,1.098901,20.000267,67037440
9,9,18,1659.4,2.444746,20.000248,58073088


,instance,algorithm,runs,distance_mean,distance_std,gap_mean,gap_std,elapsed_mean,elapsed_std,iterations_mean,iterations_std
0,RC101-100,sa,10,1647.23,10.514,1.693,0.649,20.0,0.0,63886924.8,9234386.923


In [9]:
from IPython.display import Markdown, display
table = results25
best = table.loc[table.distance.idxmin()]
display(Markdown(f"### Observations - C101-25\n\n"
    f"Across {len(table)} seeds, mean gap is {table.gap_pct.mean():.2f}% "
    f"with sample SD {table.gap_pct.std():.2f} percentage points. "
    f"Mean distance is {table.distance.mean():.1f} (SD {table.distance.std():.1f}). "
    f"Best distance: {best.distance:.1f}, seed {int(best.seed)}, "
    f"{int(best.vehicles)} trucks. Mean elapsed time: {table.elapsed_s.mean():.3f} s. "
    f"Mean SA move attempts: {table.iterations.mean():,.0f} "
    f"(SD {table.iterations.std():,.0f})."))


### Observations - C101-25

Across 10 seeds, mean gap is 0.00% with sample SD 0.00 percentage points. Mean distance is 191.3 (SD 0.0). Best distance: 191.3, seed 0, 3 trucks. Mean elapsed time: 20.000 s. Mean SA move attempts: 59,089,997 (SD 2,290,709).

In [10]:
from IPython.display import Markdown, display
table = rc_results
best = table.loc[table.distance.idxmin()]
display(Markdown(f"### Observations - RC101-100\n\n"
    f"Across {len(table)} seeds, mean gap is {table.gap_pct.mean():.2f}% "
    f"with sample SD {table.gap_pct.std():.2f} percentage points. "
    f"Mean distance is {table.distance.mean():.1f} (SD {table.distance.std():.1f}). "
    f"Best distance: {best.distance:.1f}, seed {int(best.seed)}, "
    f"{int(best.vehicles)} trucks. Mean elapsed time: {table.elapsed_s.mean():.3f} s. "
    f"Mean SA move attempts: {table.iterations.mean():,.0f} "
    f"(SD {table.iterations.std():,.0f})."))


### Observations - RC101-100

Across 10 seeds, mean gap is 1.69% with sample SD 0.65 percentage points. Mean distance is 1647.2 (SD 10.5). Best distance: 1634.2, seed 5, 16 trucks. Mean elapsed time: 20.000 s. Mean SA move attempts: 63,886,925 (SD 9,234,387).

## SA study on all 56 instances

Run seeds 0-9 on **every supplied instance**, for **560 SA runs** in total. All runs use truncated distances, the same constraints and a 20-second budget. There is no Tabu run and no instance exclusion.

`comparison` contains every instance/seed result, while `comparison_summary` reports per-instance means and sample standard deviations for distance, gap, time and iterations. Results are displayed in memory; nothing is exported automatically.

Temperature settings are fixed exploratory heuristics as explained above. Because every instance is included, interpret this as a descriptive full-dataset benchmark rather than an evaluation on a separate held-out set.


In [11]:
comparison = run_comparison(
    DATA, INSTANCES, SEEDS, time_limit=TIME_LIMIT, workers=WORKERS, algorithms=('sa',)
)
assert len(comparison) == len(INSTANCES) * len(SEEDS)
assert comparison.instance.nunique() == 56
assert comparison.algorithm.eq('sa').all()
assert comparison.groupby('instance').seed.nunique().eq(10).all()
assert comparison.feasible.all() and comparison.reference_feasible.all()
assert comparison.gap_pct.notna().all()
comparison_summary = summarize_runs(comparison)
with pd.option_context('display.max_rows', None):
    display(comparison_summary.round(3))
    display(comparison[['instance', 'seed', 'vehicles', 'distance', 'gap_pct',
                        'elapsed_s', 'iterations', 'iterations_per_s']])


,instance,algorithm,runs,distance_mean,distance_std,gap_mean,gap_std,elapsed_mean,elapsed_std,iterations_mean,iterations_std
0,C101-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,58011084.8,5234148.028
1,C102-100,sa,10,830.70,3.584,0.411,0.433,20.0,0.0,57625702.4,7524717.254
2,C103-100,sa,10,838.42,11.389,1.467,1.378,20.0,0.0,54443110.4,7852101.094
3,C104-100,sa,10,823.27,1.037,0.045,0.126,20.0,0.0,55831936.0,6706641.397
4,C105-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,59398681.6,5206203.693
5,C106-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,63293977.6,3238483.152
6,C107-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,65351808.0,2813212.860
7,C108-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,62543411.2,5097072.228
8,C109-100,sa,10,827.30,0.000,0.000,0.000,20.0,0.0,63912499.2,4507621.794
9,C201-100,sa,10,589.10,0.000,0.000,0.000,20.0,0.0,50581632.0,1448333.691


,instance,seed,vehicles,distance,gap_pct,elapsed_s,iterations,iterations_per_s
0,C101-100,0,10,827.3,0.000000e+00,20.000339,57732608,2.886581e+06
1,C101-100,1,10,827.3,2.220446e-14,20.000271,56621312,2.831027e+06
2,C101-100,2,10,827.3,2.220446e-14,20.000285,50447616,2.522345e+06
3,C101-100,3,10,827.3,2.220446e-14,20.000374,61014528,3.050669e+06
4,C101-100,4,10,827.3,2.220446e-14,20.000328,62784512,3.139174e+06
5,C101-100,5,10,827.3,2.220446e-14,20.000267,53138176,2.656873e+06
6,C101-100,6,10,827.3,-1.110223e-14,20.000339,60039168,3.001908e+06
7,C101-100,7,10,827.3,2.220446e-14,20.000322,67940096,3.396950e+06
8,C101-100,8,10,827.3,2.220446e-14,20.000284,57771008,2.888509e+06
9,C101-100,9,10,827.3,2.220446e-14,20.000553,52621824,2.631018e+06
